# Solar Spectrum Analysis Pipeline v2.0
Processing Huairou Solar Observatory spectral images (4855-4873 A region)

## Block 1: Import & Data Loading

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import os
from pathlib import Path
from astropy.io import fits
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.special import wofz
import warnings
warnings.filterwarnings('ignore')

# Version control
version = 'v2.0'

# Working directory
work_dir = Path('./')
output_dir = work_dir / f'output_{version}'
output_dir.mkdir(exist_ok=True)

print(f'Pipeline version: {version}')
print(f'Output directory: {output_dir}')

# Function to load fits images
def load_fits_images(folder_path, x_range=(0, 809)):
    '''
    Load all fits images from a folder and crop to specified x range
    '''
    images = []
    fits_files = sorted(Path(folder_path).glob('*.fits'))
    
    for fits_file in fits_files:
        with fits.open(fits_file) as hdul:
            data = hdul[0].data
            # Crop: x from 0 to 809, keep all y
            cropped = data[:, x_range[0]:x_range[1]]
            images.append(cropped)
    
    print(f'Loaded {len(images)} images from {folder_path}')
    if images:
        print(f'  Image shape: {images[0].shape}')
    
    return images

# Load images
print('\n=== Loading dark images ===')
dark_images = load_fits_images(work_dir / 'dark')

print('\n=== Loading flat images ===')
flat_images = load_fits_images(work_dir / 'flat')

print('\n=== Loading norm images ===')
norm_images = load_fits_images(work_dir / 'norm')

# Stack dark images (median)
print('\n=== Creating master dark ===')
dark_master = np.median(np.array(dark_images), axis=0)
print(f'Master dark shape: {dark_master.shape}')
print(f'Dark value range: {dark_master.min():.1f} - {dark_master.max():.1f}')

# Stack flat images (mean)
print('\n=== Creating master flat ===')
flat_master = np.mean(np.array(flat_images), axis=0)
print(f'Master flat shape: {flat_master.shape}')
print(f'Flat value range: {flat_master.min():.1f} - {flat_master.max():.1f}')

# Release memory
del dark_images, flat_images
print('\nMemory released for individual dark/flat images')

# Subtract dark from flat and norm
print('\n=== Dark correction ===')
flat_corrected = flat_master - dark_master
norm_corrected = np.array([img - dark_master for img in norm_images])
print(f'Corrected flat max: {flat_corrected.max():.1f}')
print(f'Corrected norm shape: {norm_corrected.shape}')
print(f'Corrected norm value range: {norm_corrected.min():.1f} - {norm_corrected.max():.1f}')

## Block 2: Tilt Fitting & Wavelength Calibration

In [ ]:
# Reference spectral lines for tilt and wavelength calibration
# Ni I 4857.395 (col ~100-140), Ni I 4866.277 (col ~580-620)
ref_line1 = {'wavelength': 4857.395, 'col_range': (100, 140), 'label': 'Ni I 4857.395'}
ref_line2 = {'wavelength': 4866.277, 'col_range': (580, 620), 'label': 'Ni I 4866.277'}

def find_line_center(spectrum_col, col_center_approx, search_width=5):
    '''
    Find precise center of a spectral line in 1D spectrum
    '''
    start = max(0, col_center_approx - search_width)
    end = min(len(spectrum_col), col_center_approx + search_width + 1)
    region = spectrum_col[start:end]
    offset = np.argmin(region)  # Find minimum (absorption line)
    return start + offset

def fit_tilt(image, ref_line1, ref_line2):
    '''
    Fit spectral line tilt using two reference lines
    Returns: (slope, intercept) where y_shift = slope * x + intercept
    '''
    # Extract columns and find line centers
    col1_min, col1_max = ref_line1['col_range']
    col2_min, col2_max = ref_line2['col_range']
    
    spectrum_col1 = image[:, col1_min:col1_max].mean(axis=1)
    spectrum_col2 = image[:, col2_min:col2_max].mean(axis=1)
    
    # Find centers
    center1_y = find_line_center(spectrum_col1, image.shape[0] // 2)
    center2_y = find_line_center(spectrum_col2, image.shape[0] // 2)
    
    center1_x = (col1_min + col1_max) / 2
    center2_x = (col2_min + col2_max) / 2
    
    # Linear fit
    slope = (center2_y - center1_y) / (center2_x - center1_x)
    intercept = center1_y - slope * center1_x
    
    return slope, intercept, (center1_x, center1_y, center2_x, center2_y)

# Fit tilt using master flat
print('=== Fitting spectral line tilt ===')
tilt_slope, tilt_intercept, tilt_points = fit_tilt(flat_corrected, ref_line1, ref_line2)
print(f'Tilt slope: {tilt_slope:.6f} pixels/column')
print(f'Tilt intercept: {tilt_intercept:.2f}')
print(f'Reference points: {tilt_points}')

def wavelength_calibration(col, ref_line1, ref_line2):
    '''
    Linear wavelength calibration
    Returns wavelength given column index
    '''
    col1_center = (ref_line1['col_range'][0] + ref_line1['col_range'][1]) / 2
    col2_center = (ref_line2['col_range'][0] + ref_line2['col_range'][1]) / 2
    
    wav1 = ref_line1['wavelength']
    wav2 = ref_line2['wavelength']
    
    # Linear fit: wav = a * col + b
    a = (wav2 - wav1) / (col2_center - col1_center)
    b = wav1 - a * col1_center
    
    if isinstance(col, np.ndarray):
        return a * col + b
    else:
        return a * col + b

# Wavelength calibration parameters
col1_center = (ref_line1['col_range'][0] + ref_line1['col_range'][1]) / 2
col2_center = (ref_line2['col_range'][0] + ref_line2['col_range'][1]) / 2
wav_a = (ref_line2['wavelength'] - ref_line1['wavelength']) / (col2_center - col1_center)
wav_b = ref_line1['wavelength'] - wav_a * col1_center

print(f'\n=== Wavelength calibration ===')
print(f'Wavelength coefficient: {wav_a:.6f} A/pixel')
print(f'Wavelength intercept: {wav_b:.3f} A')
print(f'Expected range: {wavelength_calibration(0, ref_line1, ref_line2):.3f} - {wavelength_calibration(809, ref_line1, ref_line2):.3f} A')

## Block 3: Tilt & Wavelength Calibration Visualization

In [ ]:
# Prepare display image (log scale for better contrast)
flat_display = np.log1p(flat_corrected)

fig, ax = plt.subplots(figsize=(14, 10), dpi=100)

# Plot flat image as background
im = ax.imshow(flat_display, cmap='gray', origin='upper', aspect='auto')
ax.set_xlabel('Column (pixel)', fontsize=12)
ax.set_ylabel('Row (pixel)', fontsize=12)
ax.set_title(f'Spectral Line Tilt & Wavelength Calibration {version}\nFlat image with overlay', fontsize=14)
plt.colorbar(im, ax=ax, label='log(Intensity)')

# Plot tilt lines (orange) on reference lines
cols = np.array([ref_line1['col_range'][0], ref_line1['col_range'][1],
                  ref_line2['col_range'][0], ref_line2['col_range'][1]])
rows_tilt = tilt_slope * cols + tilt_intercept

for i, col_range in enumerate([ref_line1['col_range'], ref_line2['col_range']]):
    cols_line = np.linspace(col_range[0], col_range[1], 100)
    rows_line = tilt_slope * cols_line + tilt_intercept
    ax.plot(cols_line, rows_line, 'C1-', linewidth=2, label='Fitted tilt' if i == 0 else '')

# Plot wavelength lines (dark blue, should be parallel to orange lines)
ncol = flat_corrected.shape[1]
nrow = flat_corrected.shape[0]

cols_grid = np.arange(ncol)
wavelengths = wavelength_calibration(cols_grid, ref_line1, ref_line2)
wav_min = wavelengths.min()
wav_max = wavelengths.max()

# Draw lines for every 1 A
for wav_target in np.arange(np.ceil(wav_min), np.floor(wav_max) + 1, 1.0):
    # Find column corresponding to this wavelength
    col_target = (wav_target - wav_b) / wav_a
    if 0 <= col_target < ncol:
        cols_line = np.linspace(col_target - 10, col_target + 10, 100)
        cols_line = np.clip(cols_line, 0, ncol - 1)
        rows_line = tilt_slope * cols_line + tilt_intercept
        rows_line = np.clip(rows_line, 0, nrow - 1)
        ax.plot(cols_line, rows_line, color='darkblue', linewidth=1.5, alpha=0.7)
        # Label every 2 A to avoid clutter
        if abs(wav_target - round(wav_target / 2) * 2) < 0.5:
            ax.text(col_target, -30, f'{wav_target:.0f}', ha='center', fontsize=9, color='darkblue')

# Mark reference lines
ax.plot([], [], 'C1-', linewidth=2, label='Tilt fit (orange)')
ax.plot([], [], color='darkblue', linewidth=1.5, label='Wavelength grid (dark blue, 1A spacing)')

ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
output_file = output_dir / f'01_tilt_wavelength_calibration_{version}.png'
plt.savefig(output_file, dpi=150, bbox_inches='tight')
print(f'Saved: {output_file}')
plt.show()

## Block 4: Extract 1D Spectra

In [ ]:
def extract_1d_spectrum(image, tilt_slope, tilt_intercept):
    '''
    Extract 1D spectrum from 2D image by aligning to fitted tilt
    and summing along the tilted direction
    '''
    spectrum_1d = []
    
    for col in range(image.shape[1]):
        row_center = tilt_slope * col + tilt_intercept
        # Simple approach: extract a few rows around the center
        row_start = max(0, int(row_center) - 2)
        row_end = min(image.shape[0], int(row_center) + 3)
        flux = image[row_start:row_end, col].sum()
        spectrum_1d.append(flux)
    
    return np.array(spectrum_1d)

# Extract 1D spectra for all norm images
print('=== Extracting 1D spectra ===')
spectra_1d = []

for i, img in enumerate(norm_corrected):
    spec = extract_1d_spectrum(img, tilt_slope, tilt_intercept)
    spectra_1d.append(spec)

spectra_1d = np.array(spectra_1d)
print(f'Extracted {len(spectra_1d)} 1D spectra')
print(f'Spectrum shape: {spectra_1d[0].shape}')

# Crude normalization: divide by 99.9% percentile
print('\n=== Normalizing spectra ===')
spectra_normalized = np.zeros_like(spectra_1d)

for i, spec in enumerate(spectra_1d):
    p999 = np.percentile(spec, 99.9)
    spectra_normalized[i] = spec / p999

print(f'Normalized spectra shape: {spectra_normalized.shape}')
print(f'Min value: {spectra_normalized.min():.4f}, Max value: {spectra_normalized.max():.4f}')

# Wavelength array
cols_array = np.arange(spectra_normalized.shape[1])
wavelengths_array = wavelength_calibration(cols_array, ref_line1, ref_line2)
print(f'\nWavelength range: {wavelengths_array[0]:.3f} - {wavelengths_array[-1]:.3f} A')

## Block 5: H-beta Line Fitting (Voigt Profile)

In [ ]:
def voigt(x, x0, sigma, gamma, intensity, continuum):
    '''
    Voigt profile (absorption line, inverted)
    '''
    z = ((x - x0) + 1j * gamma) / (sigma * np.sqrt(2))
    voigt_profile = wofz(z).real / (sigma * np.sqrt(2 * np.pi))
    return continuum - intensity * voigt_profile / voigt_profile.max()

def identify_strong_lines(spectrum, wavelengths, mask_width=3.0):
    '''
    Identify strong absorption lines to mask during H-beta fitting
    Returns boolean mask (True = valid region for fitting)
    '''
    # Find peaks in inverted spectrum (absorption lines)
    inverted = 1.0 - spectrum
    # Use prominence-based detection
    peaks, properties = find_peaks(inverted, prominence=0.1, distance=5)
    
    mask = np.ones(len(spectrum), dtype=bool)
    
    # Mask regions around detected lines
    for peak in peaks:
        wav_peak = wavelengths[peak]
        # Exclude H-beta itself (4861.342 A) with wider margin
        if 4860.5 < wav_peak < 4862.0:
            continue
        # Mask around other lines
        mask_wav = np.abs(wavelengths - wav_peak) < mask_width
        mask &= ~mask_wav
    
    return mask

def fit_halpha_voigt(spectrum, wavelengths, hbeta_wav=4861.342):
    '''
    Fit Voigt profile to H-beta line
    Returns: voigt_params, mask_used, fit_quality
    '''
    # Identify and mask strong lines
    mask = identify_strong_lines(spectrum, wavelengths)
    
    # Find H-beta center approximately
    hbeta_idx = np.argmin(np.abs(wavelengths - hbeta_wav))
    
    # Fit region around H-beta
    fit_width = 10.0  # A
    fit_mask = (np.abs(wavelengths - hbeta_wav) < fit_width) & mask
    
    if np.sum(fit_mask) < 10:
        return None, mask, None
    
    x_fit = wavelengths[fit_mask]
    y_fit = spectrum[fit_mask]
    
    # Initial parameters
    p0 = [hbeta_wav, 0.3, 1.0, 0.3, 1.0]  # x0, sigma, gamma, intensity, continuum
    
    try:
        popt, pcov = curve_fit(voigt, x_fit, y_fit, p0=p0, maxfev=5000,
                               bounds=([hbeta_wav-1, 0.1, 0.01, 0.1, 0.5],
                                      [hbeta_wav+1, 1.0, 2.0, 1.0, 1.5]))
        
        # Calculate fit quality
        y_pred = voigt(x_fit, *popt)
        residuals = y_fit - y_pred
        chi2 = np.sum(residuals**2)
        rms = np.sqrt(chi2 / len(residuals))
        
        return popt, mask, {'chi2': chi2, 'rms': rms}
    except Exception as e:
        print(f'  Fitting error: {e}')
        return None, mask, None

# Fit H-beta for all spectra
print('=== Fitting H-beta lines ===')
hbeta_fits = []
hbeta_masks = []
hbeta_quality = []

for i, spec in enumerate(spectra_normalized):
    if i % max(1, len(spectra_normalized)//10) == 0:
        print(f'Processing spectrum {i+1}/{len(spectra_normalized)}')
    
    popt, mask, quality = fit_halpha_voigt(spec, wavelengths_array)
    hbeta_fits.append(popt)
    hbeta_masks.append(mask)
    hbeta_quality.append(quality)

print(f'\nCompleted H-beta fitting for {len(spectra_normalized)} spectra')
successful_fits = sum(1 for f in hbeta_fits if f is not None)
print(f'Successful fits: {successful_fits}/{len(hbeta_fits)}')

## Block 6: Line Identification & Matching

In [ ]:
# Reference spectral lines
reference_lines = {
    4855.418: 'Ni I',
    4855.681: 'Fe I',
    4856.019: 'Ti I',
    4856.195: 'Cr II',
    4857.395: 'Ni I',
    4859.039: 'Nd II',
    4859.134: 'Fe I',
    4859.747: 'Fe I',
    4860.217: 'Cr II',
    4861.849: 'Cr I',
    4863.650: 'Fe I',
    4863.936: 'Ni I',
    4864.323: 'Cr II',
    4864.738: 'V I',
    4865.618: 'Ti II',
    4866.277: 'Ni I',
    4867.537: 'Fe I',
    4867.874: 'Co I',
    4868.263: 'Ti I',
    4869.469: 'Fe I',
    4870.136: 'Ti I'
}

match_threshold = 0.05  # A

def gaussian(x, x0, sigma, intensity, continuum):
    '''
    Gaussian profile for absorption lines
    '''
    return continuum - intensity * np.exp(-(x - x0)**2 / (2 * sigma**2))

def identify_and_fit_lines(spectrum, wavelengths, hbeta_mask):
    '''
    Identify and fit Gaussian profiles to absorption lines
    Returns list of line parameters
    '''
    # Use only valid regions (masked H-beta)
    spectrum_masked = spectrum.copy()
    spectrum_masked[~hbeta_mask] = np.nan
    
    # Remove H-beta and nearby lines
    spectrum_clean = spectrum.copy()
    spectrum_clean[~hbeta_mask] = 1.0
    
    # Find peaks in inverted spectrum
    inverted = 1.0 - spectrum_clean
    peaks, properties = find_peaks(inverted, prominence=0.05, distance=3)
    
    lines = []
    
    for peak in peaks:
        wav_peak = wavelengths[peak]
        
        # Fit Gaussian to each line
        fit_width = 1.5  # A
        fit_mask = np.abs(wavelengths - wav_peak) < fit_width
        
        if np.sum(fit_mask) < 5:
            continue
        
        x_fit = wavelengths[fit_mask]
        y_fit = spectrum_clean[fit_mask]
        
        try:
            p0 = [wav_peak, 0.3, 0.2, 1.0]
            popt, pcov = curve_fit(gaussian, x_fit, y_fit, p0=p0, maxfev=1000,
                                    bounds=([wav_peak-0.5, 0.05, 0.01, 0.7],
                                           [wav_peak+0.5, 1.0, 0.5, 1.1]))
            
            x0, sigma, intensity, continuum = popt
            
            # Calculate equivalent width (EW)
            ew = intensity * sigma * np.sqrt(2 * np.pi)
            
            # Fit quality
            y_pred = gaussian(x_fit, *popt)
            rms = np.sqrt(np.mean((y_fit - y_pred)**2))
            
            # Match to reference lines
            element = None
            delta_wav = None
            delta_v = None
            
            for ref_wav, elem in reference_lines.items():
                if abs(x0 - ref_wav) < match_threshold:
                    element = elem
                    delta_wav = x0 - ref_wav
                    delta_v = (delta_wav / ref_wav) * 3e5  # km/s
                    break
            
            lines.append({
                'wavelength_meas': x0,
                'sigma': sigma,
                'depth': intensity,
                'ew': ew,
                'rms': rms,
                'element': element,
                'delta_wav': delta_wav,
                'delta_v': delta_v,
                'reference_wav': None if element is None else [k for k, v in reference_lines.items() if v == element and abs(x0-k) < match_threshold][0]
            })
        except:
            continue
    
    return lines

# Identify lines in all spectra
print('=== Identifying spectral lines ===')
all_lines = []

for i, spec in enumerate(spectra_normalized):
    if i % max(1, len(spectra_normalized)//10) == 0:
        print(f'Processing spectrum {i+1}/{len(spectra_normalized)}')
    
    lines = identify_and_fit_lines(spec, wavelengths_array, hbeta_masks[i])
    all_lines.append(lines)

print(f'\nLine identification complete')
print(f'Average lines per spectrum: {np.mean([len(l) for l in all_lines]):.1f}')

## Block 7: Diagnostic Plots for Each Spectrum

In [ ]:
def create_spectrum_figure(spectrum, wavelengths, hbeta_fit, hbeta_mask, lines, spec_idx, version):
    '''
    Create diagnostic figure for a single spectrum
    '''
    fig = plt.figure(figsize=(14, 10), dpi=100)
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.3)
    
    # Upper panel: H-beta fitting
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(wavelengths, spectrum, 'k-', linewidth=1, label='Spectrum')
    
    # Plot masked regions (not used for H-beta fitting)
    masked_regions = np.where(~hbeta_mask)[0]
    if len(masked_regions) > 0:
        ax1.fill_between(wavelengths[~hbeta_mask], 0, 2, alpha=0.2, color='red', 
                          label='Masked regions (strong lines)')
    
    # Plot Voigt fit if successful
    if hbeta_fit is not None:
        x_fine = np.linspace(wavelengths.min(), wavelengths.max(), 1000)
        y_fine = voigt(x_fine, *hbeta_fit)
        ax1.plot(x_fine, y_fine, 'b-', linewidth=2, label='Voigt fit (H-beta)')
    
    ax1.set_ylabel('Normalized Intensity', fontsize=12)
    ax1.set_title(f'Spectrum {spec_idx+1} - H-beta Fitting {version}', fontsize=13)
    ax1.legend(loc='upper right', fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0, 1.3])
    ax1.set_xlim([wavelengths.min(), wavelengths.max()])
    
    # Lower panel: H-beta subtracted + line identification
    ax2 = fig.add_subplot(gs[1])
    
    # H-beta subtracted spectrum
    spectrum_sub = spectrum.copy()
    if hbeta_fit is not None:
        spectrum_sub = spectrum - (voigt(wavelengths, *hbeta_fit) - 1.0)
    
    ax2.plot(wavelengths, spectrum_sub, 'k-', linewidth=1, label='H-beta subtracted')
    
    # Plot identified lines
    colors = ['C1' if line['element'] is not None else 'C2' for line in lines]
    
    for i, line in enumerate(lines):
        wav = line['wavelength_meas']
        depth = line['depth']
        ax2.plot(wav, spectrum_sub[np.argmin(np.abs(wavelengths - wav))], 'o', 
                color=colors[i], markersize=6)
        
        # Label with element and offset if matched
        label_text = f'{wav:.3f}'
        if line['element'] is not None:
            delta_v_str = f"{line['delta_v']:.1f}" if line['delta_v'] is not None else 'N/A'
            label_text += f"\n{line['element']}\nDelta_v={delta_v_str}km/s"
        
        ax2.text(wav, spectrum_sub[np.argmin(np.abs(wavelengths - wav))] - 0.08, 
                label_text, ha='center', fontsize=8, rotation=0)
    
    ax2.set_xlabel('Wavelength (A)', fontsize=12)
    ax2.set_ylabel('Normalized Intensity', fontsize=12)
    ax2.set_title(f'Line Identification and Matching', fontsize=13)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 1.3])
    ax2.set_xlim([wavelengths.min(), wavelengths.max()])
    
    # Legend: matched vs unmatched
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='C1', label='Matched to reference'),
                       Patch(facecolor='C2', label='Unmatched')]
    ax2.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    return fig

# Generate figures for first few spectra and sample throughout
print('=== Generating diagnostic plots ===')
n_spectra_to_plot = min(5, len(spectra_normalized))
indices_to_plot = np.linspace(0, len(spectra_normalized)-1, n_spectra_to_plot, dtype=int)

for idx in indices_to_plot:
    fig = create_spectrum_figure(spectra_normalized[idx], wavelengths_array,
                                 hbeta_fits[idx], hbeta_masks[idx],
                                 all_lines[idx], idx, version)
    
    output_file = output_dir / f'02_spectrum_{idx:03d}_{version}.png'
    fig.savefig(output_file, dpi=150, bbox_inches='tight')
    print(f'Saved: {output_file}')
    plt.close(fig)

print(f'\nGenerated diagnostic plots for {len(indices_to_plot)} spectra')

## Block 8: Output All Spectra to FITS File

In [ ]:
def create_fits_output(spectra, wavelengths, hbeta_fits, all_lines, output_path):
    '''
    Create FITS file with all spectra and metadata for TOPCAT
    '''
    # Create primary HDU
    hdu_primary = fits.PrimaryHDU()
    hdu_primary.header['VERSION'] = version
    hdu_primary.header['COMMENT'] = 'Solar spectrum analysis pipeline output'
    hdu_primary.header['TELESCOP'] = 'Huairou Solar Observatory'
    hdu_primary.header['WAVELMIN'] = wavelengths.min()
    hdu_primary.header['WAVELMAX'] = wavelengths.max()
    hdu_primary.header['NSPEC'] = len(spectra)
    
    # Create HDU for wavelength array
    hdu_wavelength = fits.BinTableHDU.from_columns([
        fits.Column(name='WAVELENGTH', format='E', array=wavelengths)
    ])
    hdu_wavelength.header['EXTNAME'] = 'WAVELENGTH'
    
    # Create HDU for spectra
    cols = []
    for i in range(len(spectra)):
        cols.append(fits.Column(name=f'SPEC_{i:03d}', format='E', array=spectra[i]))
    hdu_spectra = fits.BinTableHDU.from_columns(cols)
    hdu_spectra.header['EXTNAME'] = 'SPECTRA'
    
    # Create HDU for line catalog
    n_lines_max = max([len(lines) for lines in all_lines]) if all_lines else 0
    
    spectrum_ids = []
    wavelengths_meas = []
    elements = []
    sigmas = []
    depths = []
    ews = []
    delta_wavs = []
    delta_vs = []
    
    for spec_idx, lines in enumerate(all_lines):
        for line in lines:
            spectrum_ids.append(spec_idx)
            wavelengths_meas.append(line['wavelength_meas'])
            elements.append(line['element'] if line['element'] else 'Unknown')
            sigmas.append(line['sigma'])
            depths.append(line['depth'])
            ews.append(line['ew'])
            delta_wavs.append(line['delta_wav'] if line['delta_wav'] else np.nan)
            delta_vs.append(line['delta_v'] if line['delta_v'] else np.nan)
    
    hdu_lines = fits.BinTableHDU.from_columns([
        fits.Column(name='SPEC_ID', format='J', array=np.array(spectrum_ids, dtype=int)),
        fits.Column(name='WAVELENGTH', format='E', array=np.array(wavelengths_meas)),
        fits.Column(name='ELEMENT', format='20A', array=np.array(elements)),
        fits.Column(name='SIGMA', format='E', array=np.array(sigmas)),
        fits.Column(name='DEPTH', format='E', array=np.array(depths)),
        fits.Column(name='EW', format='E', array=np.array(ews)),
        fits.Column(name='DELTA_WAV', format='E', array=np.array(delta_wavs)),
        fits.Column(name='DELTA_V', format='E', array=np.array(delta_vs))
    ])
    hdu_lines.header['EXTNAME'] = 'LINES'
    hdu_lines.header['COMMENT'] = 'Line catalog for all spectra'
    
    # Write to file
    hdul = fits.HDUList([hdu_primary, hdu_wavelength, hdu_spectra, hdu_lines])
    hdul.writeto(output_path, overwrite=True)
    print(f'Saved FITS file: {output_path}')
    hdul.info()

# Create output FITS file
print('=== Creating FITS output file ===')
fits_output_path = output_dir / f'solar_spectra_{version}.fits'
create_fits_output(spectra_normalized, wavelengths_array, hbeta_fits, all_lines, fits_output_path)

print(f'\n=== Pipeline Complete ===')
print(f'Output directory: {output_dir}')
print(f'Total spectra processed: {len(spectra_normalized)}')
print(f'Output files:')
for f in sorted(output_dir.glob('*')):
    size_mb = f.stat().st_size / (1024*1024)
    print(f'  - {f.name} ({size_mb:.2f} MB)')

## Summary and Statistics

In [ ]:
# Generate summary statistics
print('\n' + '='*60)
print(f'PIPELINE SUMMARY - Version {version}')
print('='*60)

print('\n1. DATA LOADING')
print(f'   Dark images stacked: median of images')
print(f'   Flat images stacked: mean of images')
print(f'   Norm images: {len(norm_corrected)} total')
print(f'   Crop region: x=0-809, y=all')
print(f'   Image shape: {norm_corrected[0].shape}')

print('\n2. TILT & WAVELENGTH CALIBRATION')
print(f'   Reference lines: Ni I 4857.395, Ni I 4866.277')
print(f'   Tilt slope: {tilt_slope:.6f} pixels/column')
print(f'   Tilt intercept: {tilt_intercept:.2f} pixels')
print(f'   Wavelength range: {wavelengths_array.min():.3f} - {wavelengths_array.max():.3f} A')
print(f'   Wavelength scale: {wav_a:.6f} A/pixel')

print('\n3. 1D SPECTRUM EXTRACTION')
print(f'   Total spectra: {len(spectra_normalized)}')
print(f'   Spectrum length: {spectra_normalized.shape[1]} pixels')
print(f'   Normalization method: 99.9% percentile')

print('\n4. H-BETA FITTING')
hbeta_success = sum(1 for f in hbeta_fits if f is not None)
print(f'   Successful fits: {hbeta_success}/{len(hbeta_fits)}')
print(f'   Profile: Voigt')
print(f'   Target wavelength: 4861.342 A')

print('\n5. LINE IDENTIFICATION')
total_lines = sum(len(lines) for lines in all_lines)
matched_lines = sum(sum(1 for line in lines if line['element'] is not None) for lines in all_lines)
print(f'   Total lines identified: {total_lines}')
print(f'   Matched to reference: {matched_lines}')
print(f'   Matching threshold: {match_threshold} A')
print(f'   Reference lines: {len(reference_lines)}')

print('\n6. OUTPUT FILES')
output_files = list(output_dir.glob('*'))
print(f'   Total files: {len(output_files)}')
total_size = sum(f.stat().st_size for f in output_files) / (1024*1024)
print(f'   Total size: {total_size:.2f} MB')

print('\n' + '='*60)